In [1]:
import numpy as np
import pandas as pd

from pathlib import Path

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "customer_purchase_propensity_dataset.parquet"
)

modeling_df = pd.read_parquet(DATA_PATH)

feature_columns = [
    "recency_days",
    "log1p_frequency",
    "log1p_monetary",
    "log1p_unique_products",
    "log1p_cancellation_value",
]

target_column = "repeat_purchase_90d"

# Reconstruct the selected Phase 9 model
train_df = modeling_df.loc[
    modeling_df["partition"] == "train"
].copy()

test_df = modeling_df.loc[
    modeling_df["partition"] == "test"
].copy()

selected_model = Pipeline([
    ("scaler", StandardScaler()),
    ("logistic_regression", LogisticRegression(
        C=0.01,
        max_iter=1000,
        random_state=42
    ))
])

selected_model.fit(
    train_df[feature_columns],
    train_df[target_column]
)

# Generate test-period predictions
test_probabilities = selected_model.predict_proba(
    test_df[feature_columns]
)[:, 1]

# Assemble the customer-snapshot prediction dataset
test_predictions_df = test_df[
    ["Customer ID", "snapshot_date"] + feature_columns + [target_column]
].copy()

test_predictions_df["predicted_repeat_probability"] = (
    test_probabilities
)

# Recover monetary value from its log1p-transformed feature
test_predictions_df["historical_monetary_180d"] = np.expm1(
    test_predictions_df["log1p_monetary"]
)

# Retain the latest available snapshot for each customer
customer_priority_df = (
    test_predictions_df
    .sort_values(
        ["Customer ID", "snapshot_date"]
    )
    .drop_duplicates(
        subset="Customer ID",
        keep="last"
    )
    .reset_index(drop=True)
)

print("Test-period customer-snapshot observations:", len(test_predictions_df))
print("Unique test-period customers:", test_predictions_df["Customer ID"].nunique())

print("\nSelected customer-level population:")
print("Observations:", len(customer_priority_df))
print("Unique customers:", customer_priority_df["Customer ID"].nunique())

print("\nSelected snapshot dates:")
print(customer_priority_df["snapshot_date"].value_counts().sort_index())

print("\nCustomer-level dataset shape:", customer_priority_df.shape)
print("\nMissing values:")
print(customer_priority_df.isna().sum())

Test-period customer-snapshot observations: 5534
Unique test-period customers: 2968

Selected customer-level population:
Observations: 2968
Unique customers: 2968

Selected snapshot dates:
snapshot_date
2011-08-01     200
2011-09-01    2768
Name: count, dtype: int64

Customer-level dataset shape: (2968, 10)

Missing values:
Customer ID                     0
snapshot_date                   0
recency_days                    0
log1p_frequency                 0
log1p_monetary                  0
log1p_unique_products           0
log1p_cancellation_value        0
repeat_purchase_90d             0
predicted_repeat_probability    0
historical_monetary_180d        0
dtype: int64


In [2]:
# Calculate initial distribution-based cutoffs
propensity_cutoff = customer_priority_df[
    "predicted_repeat_probability"
].median()

monetary_cutoff = customer_priority_df[
    "historical_monetary_180d"
].median()

print("Initial prioritization cutoffs:")
print(f"Median predicted repeat-purchase probability: {propensity_cutoff:.4f}")
print(f"Median historical monetary value (180d): {monetary_cutoff:,.2f}")

# Examine distributions
priority_dimension_summary = (
    customer_priority_df[
        [
            "predicted_repeat_probability",
            "historical_monetary_180d",
        ]
    ]
    .describe(
        percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]
    )
    .T
)

print("\nPrioritization dimension distributions:")
print(priority_dimension_summary)

Initial prioritization cutoffs:
Median predicted repeat-purchase probability: 0.4791
Median historical monetary value (180d): 454.18

Prioritization dimension distributions:
                                  count       mean        std    min      10%  \
predicted_repeat_probability 2,968.0000     0.5233     0.1975 0.1252   0.3011   
historical_monetary_180d     2,968.0000 1,197.2293 4,213.6505 2.9000 129.3000   

                                  25%      50%        75%        90%  \
predicted_repeat_probability   0.3581   0.4791     0.6792     0.8159   
historical_monetary_180d     229.3125 454.1800 1,037.2850 2,104.7620   

                                      max  
predicted_repeat_probability       0.9966  
historical_monetary_180d     118,081.9200  


In [3]:
# Assign propensity and historical-value categories
customer_priority_df["propensity_group"] = np.where(
    customer_priority_df["predicted_repeat_probability"]
    >= propensity_cutoff,
    "High propensity",
    "Low propensity"
)

customer_priority_df["value_group"] = np.where(
    customer_priority_df["historical_monetary_180d"]
    >= monetary_cutoff,
    "High historical value",
    "Low historical value"
)

# Assign priority groups
conditions = [
    (
        (customer_priority_df["value_group"] == "High historical value") &
        (customer_priority_df["propensity_group"] == "Low propensity")
    ),
    (
        (customer_priority_df["value_group"] == "High historical value") &
        (customer_priority_df["propensity_group"] == "High propensity")
    ),
    (
        (customer_priority_df["value_group"] == "Low historical value") &
        (customer_priority_df["propensity_group"] == "Low propensity")
    ),
    (
        (customer_priority_df["value_group"] == "Low historical value") &
        (customer_priority_df["propensity_group"] == "High propensity")
    ),
]

priority_labels = [
    "Priority 1 - High value, low propensity",
    "Priority 2 - High value, high propensity",
    "Priority 3 - Low value, low propensity",
    "Priority 4 - Low value, high propensity",
]

customer_priority_df["priority_group"] = np.select(
    conditions,
    priority_labels,
    default="Unassigned"
)

# Summarize customer counts and historical revenue
priority_summary = (
    customer_priority_df
    .groupby("priority_group", observed=True)
    .agg(
        customer_count=("Customer ID", "nunique"),
        median_probability=("predicted_repeat_probability", "median"),
        mean_probability=("predicted_repeat_probability", "mean"),
        median_historical_value=("historical_monetary_180d", "median"),
        mean_historical_value=("historical_monetary_180d", "mean"),
        total_historical_value=("historical_monetary_180d", "sum"),
    )
)

priority_summary["customer_share"] = (
    priority_summary["customer_count"]
    / len(customer_priority_df)
)

priority_summary["historical_revenue_share"] = (
    priority_summary["total_historical_value"]
    / customer_priority_df["historical_monetary_180d"].sum()
)

print("Customer priority group summary:")
print(priority_summary.round(4))

Customer priority group summary:
                                          customer_count  median_probability  \
priority_group                                                                 
Priority 1 - High value, low propensity              228              0.4310   
Priority 2 - High value, high propensity            1256              0.7119   
Priority 3 - Low value, low propensity              1256              0.3458   
Priority 4 - Low value, high propensity              228              0.5350   

                                          mean_probability  \
priority_group                                               
Priority 1 - High value, low propensity             0.4237   
Priority 2 - High value, high propensity            0.7164   
Priority 3 - Low value, low propensity              0.3434   
Priority 4 - Low value, high propensity             0.5495   

                                          median_historical_value  \
priority_group                              

In [4]:
priority_outcome_summary = (
    customer_priority_df
    .groupby("priority_group", observed=True)
    .agg(
        customer_count=("Customer ID", "nunique"),
        predicted_repeat_rate=(
            "predicted_repeat_probability", "mean"
        ),
        actual_repeat_rate=(target_column, "mean"),
        actual_repeat_customers=(target_column, "sum"),
        historical_monetary_value=(
            "historical_monetary_180d", "sum"
        ),
    )
)

priority_outcome_summary["actual_nonrepeat_customers"] = (
    priority_outcome_summary["customer_count"]
    - priority_outcome_summary["actual_repeat_customers"]
)

priority_outcome_summary["prediction_gap"] = (
    priority_outcome_summary["actual_repeat_rate"]
    - priority_outcome_summary["predicted_repeat_rate"]
)

priority_outcome_summary["actual_repeat_rate"] *= 100
priority_outcome_summary["predicted_repeat_rate"] *= 100
priority_outcome_summary["prediction_gap"] *= 100

print("Retrospective priority-group validation:")
print(priority_outcome_summary.round(2))

Retrospective priority-group validation:
                                          customer_count  \
priority_group                                             
Priority 1 - High value, low propensity              228   
Priority 2 - High value, high propensity            1256   
Priority 3 - Low value, low propensity              1256   
Priority 4 - Low value, high propensity              228   

                                          predicted_repeat_rate  \
priority_group                                                    
Priority 1 - High value, low propensity                 42.3700   
Priority 2 - High value, high propensity                71.6400   
Priority 3 - Low value, low propensity                  34.3400   
Priority 4 - Low value, high propensity                 54.9500   

                                          actual_repeat_rate  \
priority_group                                                 
Priority 1 - High value, low propensity              41.6700   
Pri

In [5]:
# Reconstruct frequency from log1p transformation
customer_priority_df["frequency_180d"] = np.expm1(
    customer_priority_df["log1p_frequency"]
)

snapshot_rfm_summary = (
    customer_priority_df[
        [
            "recency_days",
            "frequency_180d",
            "historical_monetary_180d",
        ]
    ]
    .describe(
        percentiles=[0.20, 0.40, 0.60, 0.80]
    )
    .T
)

print("Snapshot-specific RFM distributions:")
print(snapshot_rfm_summary.round(2))

print("\nFrequency distribution:")
print(
    customer_priority_df["frequency_180d"]
    .value_counts()
    .sort_index()
    .head(20)
)

Snapshot-specific RFM distributions:
                              count       mean        std    min      20%  \
recency_days             2,968.0000    72.1600    54.4700 0.0000  17.0000   
frequency_180d           2,968.0000     2.7000     3.9800 1.0000   1.0000   
historical_monetary_180d 2,968.0000 1,197.2300 4,213.6500 2.9000 193.3000   

                              40%      60%        80%          max  
recency_days              43.0000  82.0000   134.0000     179.0000  
frequency_180d             1.0000   2.0000     3.0000      83.0000  
historical_monetary_180d 339.5700 624.3800 1,294.4400 118,081.9200  

Frequency distribution:
frequency_180d
1.0000     1460
2.0000      602
3.0000      314
4.0000      200
5.0000      122
6.0000       69
7.0000       39
8.0000       35
9.0000       23
10.0000      17
11.0000      17
12.0000      12
13.0000       4
14.0000      10
15.0000       8
16.0000       3
17.0000       5
18.0000       2
19.0000       2
20.0000       1
Name: count, dtype

In [6]:
# Snapshot-specific RFM profile by priority group
snapshot_rfm_by_priority = (
    customer_priority_df
    .groupby("priority_group", observed=True)
    .agg(
        customer_count=("Customer ID", "nunique"),

        median_recency=("recency_days", "median"),
        mean_recency=("recency_days", "mean"),

        median_frequency=("frequency_180d", "median"),
        mean_frequency=("frequency_180d", "mean"),

        median_monetary=("historical_monetary_180d", "median"),
        mean_monetary=("historical_monetary_180d", "mean"),

        median_probability=("predicted_repeat_probability", "median"),
        actual_repeat_rate=(target_column, "mean"),
    )
)

print("Snapshot-specific RFM profile by priority group:")
print(snapshot_rfm_by_priority.round(2))

Snapshot-specific RFM profile by priority group:
                                          customer_count  median_recency  \
priority_group                                                             
Priority 1 - High value, low propensity              228        111.0000   
Priority 2 - High value, high propensity            1256         28.0000   
Priority 3 - Low value, low propensity              1256        108.0000   
Priority 4 - Low value, high propensity              228         44.5000   

                                          mean_recency  median_frequency  \
priority_group                                                             
Priority 1 - High value, low propensity       106.8800            1.0000   
Priority 2 - High value, high propensity       38.6600            3.0000   
Priority 3 - Low value, low propensity        103.3700            1.0000   
Priority 4 - Low value, high propensity        50.0500            2.0000   

                                     

In [7]:
# Define business prioritization scenarios
priority_1 = "Priority 1 - High value, low propensity"
priority_2 = "Priority 2 - High value, high propensity"
priority_3 = "Priority 3 - Low value, low propensity"
priority_4 = "Priority 4 - Low value, high propensity"

scenario_groups = {
    "A - Focused retention": [priority_1],
    "B - Broad risk coverage": [priority_1, priority_3],
    "C - High-value customer coverage": [priority_1, priority_2],
    "D - Full customer coverage": [
        priority_1, priority_2, priority_3, priority_4
    ],
}

scenario_results = []

total_customers = len(customer_priority_df)
total_historical_value = customer_priority_df[
    "historical_monetary_180d"
].sum()

for scenario_name, groups in scenario_groups.items():

    scenario_df = customer_priority_df.loc[
        customer_priority_df["priority_group"].isin(groups)
    ]

    scenario_results.append({
        "scenario": scenario_name,
        "customer_count": len(scenario_df),
        "customer_share": len(scenario_df) / total_customers,
        "historical_monetary_value": (
            scenario_df["historical_monetary_180d"].sum()
        ),
        "historical_value_share": (
            scenario_df["historical_monetary_180d"].sum()
            / total_historical_value
        ),
        "mean_predicted_probability": (
            scenario_df["predicted_repeat_probability"].mean()
        ),
        "actual_repeat_rate": scenario_df[target_column].mean(),
    })

scenario_comparison_df = pd.DataFrame(scenario_results)

print("Business prioritization scenario comparison:")
print(scenario_comparison_df.round(4).to_string(index=False))

Business prioritization scenario comparison:
                        scenario  customer_count  customer_share  historical_monetary_value  historical_value_share  mean_predicted_probability  actual_repeat_rate
           A - Focused retention             228          0.0768               159,677.4600                  0.0449                      0.4237              0.4167
         B - Broad risk coverage            1484          0.5000               435,265.0310                  0.1225                      0.3558              0.4077
C - High-value customer coverage            1484          0.5000             3,202,776.5910                  0.9013                      0.6715              0.7318
      D - Full customer coverage            2968          1.0000             3,553,376.4520                  1.0000                      0.5233              0.5822


In [8]:
# Separate customers by their observed 90-day outcome
customer_priority_df["actual_outcome"] = np.where(
    customer_priority_df[target_column] == 1,
    "Repeat purchase",
    "No repeat purchase"
)

# Historical monetary value by priority group and actual outcome
outcome_value_summary = (
    customer_priority_df
    .groupby(
        ["priority_group", "actual_outcome"],
        observed=True
    )
    .agg(
        customer_count=("Customer ID", "nunique"),
        historical_monetary_value=(
            "historical_monetary_180d", "sum"
        ),
        mean_historical_value=(
            "historical_monetary_180d", "mean"
        ),
    )
    .reset_index()
)

# Calculate each outcome's share of its priority group's historical value
outcome_value_summary["group_value_share"] = (
    outcome_value_summary["historical_monetary_value"]
    / outcome_value_summary.groupby(
        "priority_group"
    )["historical_monetary_value"].transform("sum")
)

print("Historical monetary value by priority group and outcome:")
print(outcome_value_summary.round(4).to_string(index=False))

Historical monetary value by priority group and outcome:
                          priority_group     actual_outcome  customer_count  historical_monetary_value  mean_historical_value  group_value_share
 Priority 1 - High value, low propensity No repeat purchase             133                95,168.4500               715.5523             0.5960
 Priority 1 - High value, low propensity    Repeat purchase              95                64,509.0100               679.0422             0.4040
Priority 2 - High value, high propensity No repeat purchase             265               381,684.1210             1,440.3174             0.1254
Priority 2 - High value, high propensity    Repeat purchase             991             2,661,415.0100             2,685.5853             0.8746
  Priority 3 - Low value, low propensity No repeat purchase             746               157,512.8100               211.1432             0.5716
  Priority 3 - Low value, low propensity    Repeat purchase             5